## COMPARAÇÃO DE ARQUITETURAS DE REDES NEURAIS CONVOLUCIONAIS PARA CLASSIFICAÇÃO DE CÁRIES EM RADIOGRAFIAS PANORÂMICAS

### Tarefas implementadas
- **Task 4**: Detecção de dentes cariados (binário) com Inception-v3, InceptionResNet-v2, Xception e EfficientNetV2-S

### Datasets
- **InReDD** (FD): `/content/drive/MyDrive/INREDD`
- **DENTEX** (PD): `/content/drive/MyDrive/DENTEX`

### Saída
Todos os dados processados e modelos são salvos em:
`/content/drive/MyDrive/DENTAL_PREPROCESSED/`

---

## Célula 1 — Montar Drive e Configurar Caminhos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Caminhos de entrada ─────────────────────────────────────────
INREDD_DIR   = '/content/drive/MyDrive/INREDD'
DENTEX_DIR   = '/content/drive/MyDrive/DENTEX'
OUTPUT_DIR   = '/content/drive/MyDrive/DENTAL_PREPROCESSED'

# InReDD
INREDD_IMAGES     = os.path.join(INREDD_DIR, 'images')
INREDD_FDI_JSON   = os.path.join(INREDD_DIR, 'annotations', 'teeth_fdi_labels.json')
INREDD_TEETH_JSON = os.path.join(INREDD_DIR, 'annotations', 'mouth_and_teeth_labels.json')

# DENTEX
DENTEX_TRAIN_IMG  = os.path.join(DENTEX_DIR, 'training_data',  'quadrant-enumeration-disease', 'xrays')
DENTEX_TRAIN_JSON = os.path.join(DENTEX_DIR, 'training_data',  'quadrant-enumeration-disease', 'train_quadrant_enumeration_disease.json')
DENTEX_VAL_IMG    = os.path.join(DENTEX_DIR, 'validation_data','quadrant_enumeration_disease', 'xrays')
DENTEX_VAL_JSON   = os.path.join(DENTEX_DIR, 'validation_data','quadrant_enumeration_disease', 'validation_triple.json')
DENTEX_TEST_IMG   = os.path.join(DENTEX_DIR, 'test_data', 'disease', 'input')
DENTEX_TEST_LBL   = os.path.join(DENTEX_DIR, 'test_data', 'disease', 'label')

# ── Caminhos de saída ───────────────────────────────────────────
TASK4_CAV_DIR    = os.path.join(OUTPUT_DIR, 'task4_caries', 'cavitated')
TASK4_NONCAV_DIR = os.path.join(OUTPUT_DIR, 'task4_caries', 'non_cavitated')
SPLITS_DIR       = os.path.join(OUTPUT_DIR, 'splits')
MODELS_DIR       = os.path.join(OUTPUT_DIR, 'models')

# Criar estrutura de diretórios
for d in [TASK4_CAV_DIR, TASK4_NONCAV_DIR, SPLITS_DIR,
          os.path.join(MODELS_DIR, 'task4')]:
    os.makedirs(d, exist_ok=True)

print('Diretórios criados com sucesso.')
print(f'   Saída principal: {OUTPUT_DIR}')

## Célula 2 — Instalar Dependências

In [ ]:
# Instalar dependências — inclui psutil para monitoramento de RAM
!pip install -q pandas scikit-learn pillow psutil

import tensorflow as tf
import psutil
print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')
print(f'psutil     : {psutil.__version__}')

# Verificar se monitoramento de GPU está disponível
try:
    info = tf.config.experimental.get_memory_info('GPU:0')
    print(f'GPU memory info disponível: {info}')
    GPU_MONITOR = True
except Exception as e:
    print(f'GPU memory info indisponível ({e}) — será registrado como None')
    GPU_MONITOR = False

## Célula 3 — Importações e Funções Utilitárias

In [ ]:
import json
import time
import random
import datetime
import numpy as np
import pandas as pd
import psutil
import os

from PIL import Image
from collections import Counter, defaultdict
from pathlib import Path

import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score, recall_score, accuracy_score,
    f1_score, confusion_matrix
)

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ── Constantes (idênticas ao original) ──────────────────────────
TARGET_SIZE      = (299, 299)
EXPANSION_FACTOR = 3.0
RANDOM_SEED      = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

INREDD_CAVITATED_IDS = {10, 15}
INREDD_HEALTHY_IDS   = {3}
DENTEX_CARIES_IDS    = {1, 3}


# ────────────────────────────────────────────────────────────────
#  FUNÇÕES DE MONITORAMENTO
# ────────────────────────────────────────────────────────────────

def get_ram_mb():
    """Retorna uso de RAM do processo atual em MB."""
    proc = psutil.Process(os.getpid())
    return proc.memory_info().rss / 1024 / 1024


def get_ram_system_mb():
    """Retorna uso total de RAM do sistema em MB."""
    vm = psutil.virtual_memory()
    return {
        'total_mb' : vm.total    / 1024 / 1024,
        'used_mb'  : vm.used     / 1024 / 1024,
        'available_mb': vm.available / 1024 / 1024,
        'percent'  : vm.percent
    }


def get_gpu_mb():
    """Retorna uso de VRAM da GPU em MB. Retorna None se indisponível."""
    try:
        info = tf.config.experimental.get_memory_info('GPU:0')
        return {
            'current_mb': info['current'] / 1024 / 1024,
            'peak_mb'   : info['peak']    / 1024 / 1024
        }
    except Exception:
        return None


def reset_gpu_peak():
    """Reseta o contador de pico de VRAM entre folds."""
    try:
        tf.config.experimental.reset_memory_stats('GPU:0')
    except Exception:
        pass


def fmt_duration(seconds):
    """Formata segundos em string legível HH:MM:SS."""
    return str(datetime.timedelta(seconds=int(seconds)))


# ────────────────────────────────────────────────────────────────
#  CALLBACK DE HISTÓRICO DE ÉPOCAS
# ────────────────────────────────────────────────────────────────

class EpochProfilerCallback(tf.keras.callbacks.Callback):
    """
    Registra por época:
      - accuracy e loss de treino e validação
      - learning rate atual
      - tempo de duração da época (segundos)
      - uso de RAM e GPU no final de cada época
    """
    def __init__(self):
        super().__init__()
        self.epoch_logs = []
        self._epoch_start = None

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        duration = time.time() - self._epoch_start

        # Learning rate atual
        try:
            lr = float(tf.keras.backend.get_value(
                self.model.optimizer.learning_rate
            ))
        except Exception:
            lr = None

        entry = {
            'epoch'       : epoch + 1,
            'duration_s'  : round(duration, 2),
            'train_acc'   : round(logs.get('accuracy',     0), 6),
            'train_loss'  : round(logs.get('loss',         0), 6),
            'val_acc'     : round(logs.get('val_accuracy', 0), 6),
            'val_loss'    : round(logs.get('val_loss',     0), 6),
            'lr'          : lr,
            'ram_proc_mb' : round(get_ram_mb(), 1),
            'gpu'         : get_gpu_mb()
        }
        self.epoch_logs.append(entry)


# ────────────────────────────────────────────────────────────────
#  FUNÇÕES UTILITÁRIAS
# ────────────────────────────────────────────────────────────────

def expand_bbox(x, y, w, h, img_w, img_h, factor=EXPANSION_FACTOR):
    cx = x + w / 2;  cy = y + h / 2
    new_w = w * factor;  new_h = h * factor
    x1 = max(0,     int(cx - new_w / 2))
    y1 = max(0,     int(cy - new_h / 2))
    x2 = min(img_w, int(cx + new_w / 2))
    y2 = min(img_h, int(cy + new_h / 2))
    return x1, y1, x2, y2

def polygon_to_bbox(segmentation):
    coords = segmentation[0]
    xs = coords[0::2];  ys = coords[1::2]
    x, y = min(xs), min(ys)
    return x, y, max(xs) - x, max(ys) - y

def crop_and_resize(img_path, bbox_xyxy):
    try:
        img = Image.open(img_path).convert('RGB')
        x1, y1, x2, y2 = bbox_xyxy
        if x2 <= x1 or y2 <= y1: return None
        return img.crop((x1, y1, x2, y2)).resize(TARGET_SIZE, Image.LANCZOS)
    except Exception as e:
        print(f'  [Erro] {img_path}: {e}')
        return None

def dentex_fdi(cat_id_1, cat_id_2):
    return (cat_id_1 + 1) * 10 + (cat_id_2 + 1)

def save_crop(img_path, x, y, w, h, img_w, img_h, out_path):
    if os.path.exists(out_path): return True
    x1, y1, x2, y2 = expand_bbox(x, y, w, h, img_w, img_h)
    cropped = crop_and_resize(img_path, (x1, y1, x2, y2))
    if cropped is None: return False
    cropped.save(out_path, 'JPEG', quality=95)
    return True

print('Importações, callbacks e funções de monitoramento carregados.')

## Célula 4 — Task 4: Pré-processamento InReDD (Cáries)

Fonte: `mouth_and_teeth_labels.json`  
Formato: segmentação poligonal → bbox calculado  
- Cariados: `C` (id=10) + `Dc` (id=15)  
- Saudáveis: `H` (id=3)

In [ ]:
print('Carregando mouth_and_teeth_labels.json ...')
with open(INREDD_TEETH_JSON) as f:
    teeth_data = json.load(f)

teeth_img_map = {img['id']: img for img in teeth_data['images']}

caries_anns  = [a for a in teeth_data['annotations'] if a['category_id'] in INREDD_CAVITATED_IDS]
healthy_anns = [a for a in teeth_data['annotations'] if a['category_id'] in INREDD_HEALTHY_IDS]

print(f'  Total anotações: {len(teeth_data["annotations"])}')
print(f'  Cariados (C+Dc): {len(caries_anns)}')
print(f'  Saudáveis  (H) : {len(healthy_anns)}')
print('    Artigo reporta 885 cariados InReDD; dataset atual tem',
      len(caries_anns), '(versão do dataset pode diferir).')


def process_inredd_task4(annotations, label, out_dir, prefix):
    """Processa anotações InReDD para Task 4. Retorna lista de manifesto."""
    manifest = []
    skipped  = 0
    img_w_default, img_h_default = 2903, 1536

    for ann in annotations:
        img_meta = teeth_img_map.get(ann['image_id'])
        if img_meta is None:
            skipped += 1; continue

        img_path = os.path.join(INREDD_IMAGES, img_meta['file_name'])
        if not os.path.exists(img_path):
            skipped += 1; continue

        img_w = img_meta.get('width',  img_w_default)
        img_h = img_meta.get('height', img_h_default)

        seg = ann.get('segmentation')
        if not seg:
            skipped += 1; continue

        x, y, w, h = polygon_to_bbox(seg)

        # Filtrar regiões que abrangem > 50% da imagem (são regiões, não dentes)
        if w > img_w * 0.5 or h > img_h * 0.5:
            skipped += 1; continue

        out_name = f"{prefix}_{ann['image_id']}_{ann['id']}.jpg"
        out_path = os.path.join(out_dir, out_name)

        if save_crop(img_path, x, y, w, h, img_w, img_h, out_path):
            manifest.append({'path': out_path, 'label': label, 'source': 'inredd'})
        else:
            skipped += 1

    print(f'  {label}: {len(manifest)} salvos, {skipped} ignorados')
    return manifest


print('\n  Processando cariados (InReDD) ...')
inredd_cav_manifest = process_inredd_task4(
    caries_anns, 'cavitated', TASK4_CAV_DIR, 'inredd_cav'
)

print('  Processando saudáveis (InReDD) ...')
inredd_healthy_manifest = process_inredd_task4(
    healthy_anns, 'non_cavitated', TASK4_NONCAV_DIR, 'inredd_healthy'
)

## Célula 5 — Task 4: Pré-processamento DENTEX (Cáries)

Fontes utilizadas:
- `training_data/quadrant-enumeration-disease/` → JSON COCO triplo (quadrante + FDI + doença)
- `validation_data/quadrant_enumeration_disease/` → mesmo formato
- `test_data/disease/label/` → JSONs individuais (formato LabelMe)

Cariados: `category_id_3` ∈ {1=Caries, 3=Deep Caries}

In [ ]:
def process_dentex_coco(json_path, img_dir, out_dir, prefix):
    """
    Processa JSON DENTEX no formato triplo (quadrant_enumeration_disease).
    Retorna manifesto com entradas cavitated.
    """
    with open(json_path) as f:
        data = json.load(f)

    img_map  = {img['id']: img for img in data['images']}
    manifest = []
    skipped  = 0

    for ann in data['annotations']:
        # Filtrar apenas anotações de cárie
        if ann.get('category_id_3') not in DENTEX_CARIES_IDS:
            skipped += 1; continue

        img_meta = img_map.get(ann['image_id'])
        if img_meta is None:
            skipped += 1; continue

        img_path = os.path.join(img_dir, img_meta['file_name'])
        if not os.path.exists(img_path):
            skipped += 1; continue

        img_w = img_meta['width']
        img_h = img_meta['height']
        x, y, w, h = ann['bbox']
        fdi = dentex_fdi(ann['category_id_1'], ann['category_id_2'])

        out_name = f"{prefix}_{ann['image_id']}_{ann['id']}_fdi{fdi}.jpg"
        out_path = os.path.join(out_dir, out_name)

        if save_crop(img_path, x, y, w, h, img_w, img_h, out_path):
            manifest.append({
                'path': out_path, 'label': 'cavitated',
                'source': 'dentex', 'fdi': fdi
            })
        else:
            skipped += 1

    print(f'  {os.path.basename(json_path)}: {len(manifest)} cariados, {skipped} ignorados')
    return manifest


def process_dentex_test(label_dir, img_dir, out_dir):
    """
    Processa JSONs individuais do test_data DENTEX (formato LabelMe).
    Label: 'Q-condition-FDI' (ex: '1-çürük-26'). 'çürük' = cárie em turco.
    """
    CARIES_CONDITIONS = {'çürük', 'caries', 'carie', 'decay'}
    manifest = []
    skipped  = 0

    for fname in sorted(os.listdir(label_dir)):
        if not fname.endswith('.json'):
            continue
        with open(os.path.join(label_dir, fname)) as f:
            ldata = json.load(f)

        img_path = os.path.join(img_dir, ldata['imagePath'])
        if not os.path.exists(img_path):
            continue

        img_w = ldata['imageWidth']
        img_h = ldata['imageHeight']
        base  = Path(ldata['imagePath']).stem

        for i, shape in enumerate(ldata.get('shapes', [])):
            parts = shape['label'].split('-')
            if len(parts) < 2:
                continue
            condition = parts[1].lower()
            if condition not in CARIES_CONDITIONS:
                skipped += 1; continue

            pts = shape['points']   # [[x,y], ...]
            xs  = [p[0] for p in pts]
            ys  = [p[1] for p in pts]
            x, y = min(xs), min(ys)
            w = max(xs) - x
            h = max(ys) - y

            out_name = f"dentex_test_{base}_{i}.jpg"
            out_path = os.path.join(out_dir, out_name)

            if save_crop(img_path, x, y, w, h, img_w, img_h, out_path):
                manifest.append({
                    'path': out_path, 'label': 'cavitated', 'source': 'dentex_test'
                })
            else:
                skipped += 1

    print(f'  test_data: {len(manifest)} cariados, {skipped} ignorados')
    return manifest


print('  Processando DENTEX training ...')
dentex_train_manifest = process_dentex_coco(
    DENTEX_TRAIN_JSON, DENTEX_TRAIN_IMG, TASK4_CAV_DIR, 'dentex_train'
)

print('  Processando DENTEX validation ...')
dentex_val_manifest = process_dentex_coco(
    DENTEX_VAL_JSON, DENTEX_VAL_IMG, TASK4_CAV_DIR, 'dentex_val'
)

print('  Processando DENTEX test ...')
dentex_test_manifest = process_dentex_test(
    DENTEX_TEST_LBL, DENTEX_TEST_IMG, TASK4_CAV_DIR
)

all_dentex = dentex_train_manifest + dentex_val_manifest + dentex_test_manifest
print(f'\n Total DENTEX cariados: {len(all_dentex)}')

## Célula 6 — Task 4: Combinar Datasets e Balancear Classes

3.652 cariados + 3.652 não-cariados = 7.304 imagens totais

In [ ]:
random.seed(RANDOM_SEED)

# Combinar todos os cariados
all_cavitated = inredd_cav_manifest + all_dentex
print(f'Cariados disponíveis: {len(all_cavitated)}')
print(f'  InReDD: {len(inredd_cav_manifest)}')
print(f'  DENTEX: {len(all_dentex)}')

# Target: máximo possível até 3652
TARGET = min(len(all_cavitated), 3652)

random.shuffle(all_cavitated)
all_cavitated = all_cavitated[:TARGET]

# Amostrar saudáveis para igualar
random.shuffle(inredd_healthy_manifest)
all_non_cavitated = inredd_healthy_manifest[:TARGET]

print(f'\nApós balanceamento:')
print(f'  Cariados    : {len(all_cavitated)}')
print(f'  Não-cariados: {len(all_non_cavitated)}')

if TARGET < 3652:
    print(f'     Disponível nesta versão do dataset: {TARGET} por classe.')

# Montar manifesto completo
task4_manifest = all_cavitated + all_non_cavitated
random.shuffle(task4_manifest)

t4_manifest_path = os.path.join(SPLITS_DIR, 'task4_manifest.json')
with open(t4_manifest_path, 'w') as f:
    json.dump(task4_manifest, f)

print(f'\n Task 4: {len(task4_manifest)} imagens totais')
print(f' Manifesto salvo: {t4_manifest_path}')

## Célula 7 — Criar Splits de Validação Cruzada (5-Fold)

Implementação:
- `StratifiedKFold(n_splits=5)` mantém distribuição de classes
- Cada fold: FT → val (50%) + test (50%); restante → train

In [ ]:
"""
CÉLULA 7 — REFATORADA
Cria splits estratificados 80% treino / 20% validação via StratifiedKFold(n_splits=5).
Sem subdivisão adicional em conjunto de teste.

Protocolo metodológico:
  - Cada fold k usa 4/5 do dataset como treino e 1/5 como validação.
  - O modelo é avaliado sobre o conjunto de validação de cada fold.
  - Não há conjunto de teste separado durante a validação cruzada.
"""

def create_5fold_splits(manifest, prefix, splits_dir, random_seed=42):
    """
    Cria 5 splits estratificados (80% treino / 20% validação) e salva
    cada fold como JSON.

    Cada fold_k.json contém:
        {
            'fold':  int,
            'train': [ {'path': ..., 'label': ..., ...}, ... ],
            'val':   [ {'path': ..., 'label': ..., ...}, ... ]
        }

    Não há chave 'test': o fold de validação é integralmente utilizado
    para avaliação, conforme protocolo metodológico acordado.

    Parâmetros
    ----------
    manifest   : list[dict]  — lista completa de amostras com chave 'label'
    prefix     : str         — prefixo do nome dos arquivos (ex: 'task4')
    splits_dir : str         — diretório de saída dos JSONs
    random_seed: int         — semente para reprodutibilidade
    """
    items  = manifest.copy()
    labels = [item['label'] for item in items]

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(items, labels), start=1):

        fold_data = {
            'fold' : fold_idx,
            'train': [items[i] for i in train_idx],
            'val'  : [items[i] for i in val_idx],
        }

        fold_path = os.path.join(splits_dir, f'{prefix}_fold{fold_idx}.json')
        with open(fold_path, 'w') as f:
            json.dump(fold_data, f)

        tr_dist  = Counter(items[i]['label'] for i in train_idx)
        val_dist = Counter(items[i]['label'] for i in val_idx)

        tr_pct  = len(train_idx) / len(items) * 100
        val_pct = len(val_idx)   / len(items) * 100

        print(
            f'  Fold {fold_idx}: '
            f'train={len(train_idx)} ({tr_pct:.1f}%) {dict(tr_dist)}  '
            f'val={len(val_idx)} ({val_pct:.1f}%) {dict(val_dist)}'
        )

    print(f'\n  Splits salvos: {prefix}_fold1..5.json em {splits_dir}')


# ── Execução ─────────────────────────────────────────────────────────────────
print('\n Criando splits Task 4 (Cáries) ...')
with open(os.path.join(SPLITS_DIR, 'task4_manifest.json')) as f:
    t4_manifest = json.load(f)

create_5fold_splits(
    manifest    = t4_manifest,
    prefix      = 'task4',
    splits_dir  = SPLITS_DIR,
    random_seed = RANDOM_SEED,
)

print('\n Todos os splits salvos no Drive.')

### Conferir a distribuição de classes por fold (train / val)

In [ ]:
"""
CÉLULA 7b — Conferir distribuição de classes por fold (train / val)
Sem coluna 'test': protocolo 80/20 sem separação adicional.
"""

print('Distribuição de classes por fold (train / val)\n')
print(f'{"Fold":<6} {"Train Cav":>12} {"Train Non":>12} {"Val Cav":>10} {"Val Non":>10}')
print('─' * 54)

for fold in range(1, 6):
    with open(os.path.join(SPLITS_DIR, f'task4_fold{fold}.json')) as f:
        fd = json.load(f)

    def count(items):
        c = Counter(i['label'] for i in items)
        return c.get('cavitated', 0), c.get('non_cavitated', 0)

    tr_c, tr_n = count(fd['train'])
    vl_c, vl_n = count(fd['val'])

    total = tr_c + tr_n + vl_c + vl_n
    tr_pct  = (tr_c + tr_n) / total * 100
    val_pct = (vl_c + vl_n) / total * 100

    print(
        f'{fold:<6} {tr_c:>12} {tr_n:>12} {vl_c:>10} {vl_n:>10}  '
        f'[{tr_pct:.1f}% treino / {val_pct:.1f}% val]'
    )

print('\nProporções esperadas: ~80% treino / ~20% validação por fold.')

In [ ]:
"""
CÉLULA DE VERIFICAÇÃO — Integridade dos Splits e Rastreabilidade

Executa 6 verificações em sequência:
  1. Existência dos 5 arquivos JSON de folds
  2. Estrutura esperada de cada JSON (chaves 'fold', 'train', 'val')
  3. Proporções corretas (80% treino / 20% validação por fold)
  4. Equilíbrio de classes em treino e validação
  5. Correspondência entre caminhos nos JSONs e arquivos no disco
  6. Ausência de duplicatas dentro de cada fold e entre splits

Saída:
  - Relatório de texto por verificação
  - Resumo final: PASSOU / FALHOU com contagem de problemas
"""

# ── Parâmetros ────────────────────────────────────────────────────────────────
EXPECTED_FOLDS  = 5
TRAIN_PCT_MIN   = 0.78   # tolerância: ±2 pp
TRAIN_PCT_MAX   = 0.82
VAL_PCT_MIN     = 0.18
VAL_PCT_MAX     = 0.22
# Para classificação binária balanceada: proporção por classe deve ser 0.5 ± tolerância
CLASS_BALANCE_TOL = 0.03

issues = []  # acumula todos os problemas encontrados


def section(title):
    print(f'\n{"─"*60}')
    print(f'  {title}')
    print(f'{"─"*60}')


# ── 1. Existência dos arquivos JSON ───────────────────────────────────────────
section('1. Existência dos arquivos JSON dos folds')

fold_files = {}
for k in range(1, EXPECTED_FOLDS + 1):
    path = os.path.join(SPLITS_DIR, f'task4_fold{k}.json')
    exists = os.path.exists(path)
    status = 'OK' if exists else 'AUSENTE'
    print(f'  task4_fold{k}.json : {status}')
    if exists:
        fold_files[k] = path
    else:
        issues.append(f'JSON ausente: task4_fold{k}.json')

print(f'\n  {len(fold_files)}/{EXPECTED_FOLDS} arquivos encontrados.')


# ── 2. Estrutura dos JSONs ────────────────────────────────────────────────────
section('2. Estrutura dos arquivos JSON')

REQUIRED_KEYS = {'fold', 'train', 'val'}
fold_data = {}

for k, path in fold_files.items():
    with open(path) as f:
        fd = json.load(f)

    missing_keys = REQUIRED_KEYS - set(fd.keys())
    extra_keys   = set(fd.keys()) - REQUIRED_KEYS - {'test'}  # 'test' é legado; alertar

    if missing_keys:
        msg = f'Fold {k}: chaves faltando {missing_keys}'
        print(f'  [ERRO] {msg}')
        issues.append(msg)
    else:
        print(f'  Fold {k}: estrutura OK  (chaves: {set(fd.keys())})')
        fold_data[k] = fd

    if 'test' in fd:
        msg = f'Fold {k}: chave "test" presente — protocolo antigo detectado'
        print(f'  [AVISO] {msg}')
        issues.append(msg)


# ── 3. Proporções treino / validação ─────────────────────────────────────────
section('3. Proporções treino / validação por fold')

print(f'  Esperado: treino {TRAIN_PCT_MIN*100:.0f}–{TRAIN_PCT_MAX*100:.0f}% '
      f'/ val {VAL_PCT_MIN*100:.0f}–{VAL_PCT_MAX*100:.0f}%\n')

all_items_per_fold = {}

for k, fd in fold_data.items():
    n_train = len(fd['train'])
    n_val   = len(fd['val'])
    total   = n_train + n_val

    tr_pct  = n_train / total
    val_pct = n_val   / total

    tr_ok  = TRAIN_PCT_MIN <= tr_pct <= TRAIN_PCT_MAX
    val_ok = VAL_PCT_MIN   <= val_pct <= VAL_PCT_MAX

    status = 'OK' if (tr_ok and val_ok) else 'FORA DO INTERVALO'

    print(
        f'  Fold {k}: total={total}  '
        f'train={n_train} ({tr_pct*100:.1f}%)  '
        f'val={n_val} ({val_pct*100:.1f}%)  [{status}]'
    )

    if not tr_ok or not val_ok:
        msg = f'Fold {k}: proporção fora do esperado (train={tr_pct:.3f}, val={val_pct:.3f})'
        issues.append(msg)

    all_items_per_fold[k] = {'train': fd['train'], 'val': fd['val']}


# ── 4. Equilíbrio de classes ──────────────────────────────────────────────────
section('4. Equilíbrio de classes por fold')

print(f'  Tolerância por classe: 0.5 ± {CLASS_BALANCE_TOL}\n')

for k, splits in all_items_per_fold.items():
    for split_name, items in splits.items():
        counts = Counter(i['label'] for i in items)
        total  = sum(counts.values())

        if total == 0:
            issues.append(f'Fold {k} / {split_name}: split vazio')
            print(f'  [ERRO] Fold {k} / {split_name}: vazio')
            continue

        labels_found = list(counts.keys())
        proportions  = {lbl: counts[lbl] / total for lbl in labels_found}

        ok = all(
            abs(p - 0.5) <= CLASS_BALANCE_TOL
            for p in proportions.values()
        )

        status = 'OK' if ok else 'DESBALANCEADO'
        print(
            f'  Fold {k} / {split_name:5s}: '
            + '  '.join(f'{lbl}={counts[lbl]} ({p*100:.1f}%)'
                        for lbl, p in sorted(proportions.items()))
            + f'  [{status}]'
        )

        if not ok:
            msg = f'Fold {k} / {split_name}: desbalanceamento detectado {dict(proportions)}'
            issues.append(msg)


# ── 5. Correspondência caminhos x arquivos no disco ───────────────────────────
section('5. Correspondência caminhos x arquivos no disco')

missing_files  = []
found_total    = 0
missing_total  = 0

# Coleta todos os caminhos únicos entre todos os folds para evitar verificação duplicada
all_paths = set()
for k, splits in all_items_per_fold.items():
    for items in splits.values():
        for item in items:
            all_paths.add(item['path'])

print(f'  Caminhos únicos a verificar: {len(all_paths)}')

for path in all_paths:
    if os.path.exists(path):
        found_total += 1
    else:
        missing_total += 1
        missing_files.append(path)

print(f'  Encontrados no disco : {found_total}')
print(f'  Ausentes no disco    : {missing_total}')

if missing_files:
    print(f'\n  Primeiros 10 ausentes:')
    for p in missing_files[:10]:
        print(f'    {p}')
    msg = f'{missing_total} arquivos referenciados nos JSONs não existem no disco'
    issues.append(msg)
else:
    print('  Todos os arquivos encontrados no disco.')


# ── 6. Duplicatas ─────────────────────────────────────────────────────────────
section('6. Detecção de duplicatas')

dup_issues = 0

for k, splits in all_items_per_fold.items():
    # 6a. Duplicatas dentro de train
    train_paths = [i['path'] for i in splits['train']]
    train_dupes = {p for p, c in Counter(train_paths).items() if c > 1}
    if train_dupes:
        msg = f'Fold {k} / train: {len(train_dupes)} caminhos duplicados'
        print(f'  [ERRO] {msg}')
        issues.append(msg)
        dup_issues += 1

    # 6b. Duplicatas dentro de val
    val_paths = [i['path'] for i in splits['val']]
    val_dupes = {p for p, c in Counter(val_paths).items() if c > 1}
    if val_dupes:
        msg = f'Fold {k} / val: {len(val_dupes)} caminhos duplicados'
        print(f'  [ERRO] {msg}')
        issues.append(msg)
        dup_issues += 1

    # 6c. Vazamento: mesma imagem em train e val do mesmo fold
    train_set = set(train_paths)
    val_set   = set(val_paths)
    leak      = train_set & val_set
    if leak:
        msg = f'Fold {k}: {len(leak)} imagens presentes em TRAIN e VAL (data leakage)'
        print(f'  [ERRO] {msg}')
        issues.append(msg)
        dup_issues += 1

if dup_issues == 0:
    print('  Sem duplicatas ou vazamento de dados detectados.')

# 6d. Verificação cross-fold: mesma imagem no val de dois folds distintos
# (esperado em KFold — cada amostra aparece como val exatamente 1 vez)
val_fold_count = defaultdict(list)
for k, splits in all_items_per_fold.items():
    for item in splits['val']:
        val_fold_count[item['path']].append(k)

appeared_in_val = [folds for folds in val_fold_count.values() if len(folds) > 1]
if appeared_in_val:
    print(
        f'\n  [AVISO] {len(appeared_in_val)} amostras aparecem no val de mais de um fold '
        f'(inesperado em KFold puro — verifique se o manifesto contém duplicatas).'
    )
    issues.append(f'{len(appeared_in_val)} amostras no val de múltiplos folds')
else:
    print(
        '  KFold íntegro: cada amostra aparece no val de exatamente 1 fold.'
    )


# ── Relatório Final ───────────────────────────────────────────────────────────
section('RESULTADO FINAL')

if not issues:
    print('\n  PASSOU — Nenhum problema encontrado.')
    print('  Os splits estão metodologicamente corretos e prontos para uso.')
else:
    print(f'\n  FALHOU — {len(issues)} problema(s) detectado(s):\n')
    for i, msg in enumerate(issues, 1):
        print(f'    {i:2d}. {msg}')
    print('\n  Ação requerida: corrija os problemas acima antes de iniciar os treinamentos.')

## Célula 8 — Construção dos Modelos

Transfer learning com pesos ImageNet → fine-tuning completo.

| Parâmetro     | Inception-v3 | InceptionResNet-v2 | Xception | EfficientNetV2-S |
| ------------- | ------------ | ------------------ | -------- | ---------------- |
| Epochs máx    | 50           | 50                 | 50       | 50               |
| Batch size    | 32            | 32                  | 32        | 32                |
| Dropout       | 0.2          | 0.2                | 0.2      | 0.2              |
| Learning rate | 0.001        | 0.001              | 0.001    | 0.001            |


In [ ]:
def build_model(architecture, num_classes, dropout_rate):
    """
    Constrói Inception-v3, InceptionResNetV2, Xception e Efficientnetv2s com transfer learning.
    Camadas base desbloqueadas para fine-tuning completo.
    """
    input_shape = (299, 299, 3)

    if architecture == 'inceptionv3':
        base = tf.keras.applications.InceptionV3(
            weights='imagenet', include_top=False, input_shape=input_shape
        )
    elif architecture == 'inceptionresnetv2':
        base = tf.keras.applications.InceptionResNetV2(
            weights='imagenet', include_top=False, input_shape=input_shape
        )
    elif architecture == 'xception':
        base = tf.keras.applications.Xception(
            weights='imagenet', include_top=False, input_shape=input_shape
        )
    elif architecture == 'efficientnetv2s':
        base = tf.keras.applications.EfficientNetV2S(
            weights='imagenet', include_top=False, input_shape=input_shape
        )
    else:
        raise ValueError(f'Arquitetura desconhecida: {architecture}')

    base.trainable = True  # Fine-tuning completo

    x = base.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    if num_classes == 2:
        output = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    else:
        output = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inputs=base.input, outputs=output)


def get_callbacks(monitor='val_loss'):
    """
    - EarlyStopping: para se val_loss não melhorar em 6 epochs consecutivas
    - ReduceLROnPlateau: divide lr por 2 se val_loss não melhorar em 1 epoch
    """
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor=monitor, patience=6,
            restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, factor=0.5,
            patience=1, verbose=1, min_lr=1e-7
        )
    ]


print('Funções de modelo e callbacks definidas.')

## Célula 9 — Pipeline de Treinamento com ImageDataGenerator

Parâmetros de augmentação:
- `width_shift_range=0.25` (25% deslocamento horizontal)
- `height_shift_range=0.25` (25% deslocamento vertical)
- `zoom_range=0.15` (variação de zoom de 15%)
- `horizontal_flip=True` e `vertical_flip=True`
- `brightness_range=[0.8, 1.2]` (variação de brilho)
- `fill_mode='nearest'`

In [ ]:
# Augmentation generator (treino)
TRAIN_DATAGEN = ImageDataGenerator(
    rescale=1. / 255,
    width_shift_range=0.25,
    height_shift_range=0.25,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Apenas normalização (val)
TEST_DATAGEN = ImageDataGenerator(rescale=1. / 255)


def make_generator(datagen, items, batch_size, class_mode, label_map=None, shuffle=True):
    """
    Cria generator via flow_from_dataframe.
    label_map: dict {label_str: int} para garantir consistência entre folds.
    """
    df = pd.DataFrame(items)

    if class_mode == 'binary':
        label_map = label_map or {'cavitated': '1', 'non_cavitated': '0'}
        df['label_enc'] = df['label'].map(label_map)
        y_col      = 'label_enc'
        class_mode_gen = 'raw'  # 0.0 / 1.0 para binary crossentropy
        df['label_enc'] = df['label_enc'].astype(float)
    else:
        # Multi-class: usa label string diretamente
        y_col      = 'label'
        class_mode_gen = 'sparse'   # inteiros para sparse_categorical_crossentropy

    gen = datagen.flow_from_dataframe(
        dataframe=df,
        x_col='path',
        y_col=y_col,
        target_size=(299, 299),
        color_mode='rgb',
        class_mode=class_mode_gen,
        batch_size=batch_size,
        shuffle=shuffle,
        seed=RANDOM_SEED
    )
    return gen


print('ImageDataGenerator configurado.')

## Célula 10 — Treinamento Task 4: Detecção de Cáries (5-Fold)

| Parâmetro     | Inception-v3 | InceptionResNet-v2 | Xception | EfficientNetV2-S |
| ------------- | ------------ | ------------------ | -------- | ---------------- |
| Epochs máx    | 50           | 50                 | 50       | 50               |
| Batch size    | 32            | 32                  | 32       | 32                |
| Dropout       | 0.2          | 0.2                | 0.2      | 0.2              |
| Learning rate | 0.001        | 0.001              | 0.001    | 0.001            |


Classificação binária: cavitated (1) vs non_cavitated (0)

In [ ]:
"""
CÉLULA 10 — REFATORADA
Treinamento Task 4 com protocolo 80% treino / 20% validação.

Mudanças em relação à versão anterior:
  1. Removido 'test_gen': avaliação é feita sobre 'val' (fold completo).
  2. fd['test'] não existe mais nos JSONs; todo fd['val'] é usado.
  3. Profiling atualizado: n_test_images removido; n_val_images reflete 20%.
  4. Avaliação ao final de cada fold usa o mesmo val_gen com shuffle=False.
"""

TASK4_CONFIG = {
    'inceptionv3'      : {'dropout': 0.2, 'lr': 0.001, 'epochs': 50, 'batch_size': 32},
    'inceptionresnetv2': {'dropout': 0.2, 'lr': 0.001, 'epochs': 50, 'batch_size': 32},
    'xception'         : {'dropout': 0.2, 'lr': 0.001, 'epochs': 50, 'batch_size': 32},
    'efficientnetv2s'  : {'dropout': 0.2, 'lr': 0.001, 'epochs': 50, 'batch_size': 32},
}

T4_LABEL_MAP = {'cavitated': '1', 'non_cavitated': '0'}


def train_task4(architecture):
    cfg       = TASK4_CONFIG[architecture]
    model_dir = os.path.join(MODELS_DIR, 'task4')

    results   = []   # métricas de classificação
    profiling = []   # métricas de tempo e memória por fold
    history   = []   # histórico de épocas por fold

    print(f'\n{"="*60}')
    print(f'TASK 4 | {architecture.upper()} | 5-Fold CV (80/20)')
    print(f'  dropout={cfg["dropout"]}  lr={cfg["lr"]}  '
          f'epochs≤{cfg["epochs"]}  batch={cfg["batch_size"]}')
    print(f'{"="*60}')

    ram_sys_start  = get_ram_system_mb()
    model_start_ts = datetime.datetime.now().isoformat()
    model_start_t  = time.time()

    for fold in range(1, 6):
        print(f'\n--- Fold {fold}/5 ---')

        reset_gpu_peak()
        ram_before = get_ram_mb()
        gpu_before = get_gpu_mb()
        fold_start = time.time()

        fold_path = os.path.join(SPLITS_DIR, f'task4_fold{fold}.json')
        with open(fold_path) as f:
            fd = json.load(f)

        # ── Geradores ──────────────────────────────────────────────────────────
        train_gen = make_generator(
            TRAIN_DATAGEN, fd['train'],
            cfg['batch_size'], class_mode='binary',
            label_map=T4_LABEL_MAP, shuffle=True
        )
        # val_gen é usado tanto no model.fit (monitoramento de callbacks)
        # quanto na avaliação final do fold — sem separação adicional.
        val_gen = make_generator(
            TEST_DATAGEN, fd['val'],
            cfg['batch_size'], class_mode='binary',
            label_map=T4_LABEL_MAP, shuffle=False
        )

        # ── Modelo ────────────────────────────────────────────────────────────
        model = build_model(
            architecture, num_classes=2, dropout_rate=cfg['dropout']
        )
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['lr']),
            loss='binary_crossentropy',
            metrics=['accuracy'],
        )

        ckpt_path      = os.path.join(model_dir, f'{architecture}_fold{fold}.h5')
        epoch_profiler = EpochProfilerCallback()

        cbs = get_callbacks() + [
            tf.keras.callbacks.ModelCheckpoint(
                ckpt_path, save_best_only=True,
                monitor='val_loss', verbose=0
            ),
            epoch_profiler,
        ]

        # ── Treinamento ───────────────────────────────────────────────────────
        train_start = time.time()
        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=cfg['epochs'],
            callbacks=cbs,
            verbose=1,
        )
        train_duration = time.time() - train_start

        ram_after_train = get_ram_mb()
        gpu_after_train = get_gpu_mb()
        n_epochs_run    = len(epoch_profiler.epoch_logs)
        best_epoch      = int(np.argmin(
            [e['val_loss'] for e in epoch_profiler.epoch_logs]
        )) + 1

        # ── Avaliação (sobre val completo = 20% do fold) ─────────────────────
        # O val_gen precisa ser recriado para garantir estado inicial limpo
        # (o gerador pode ter sido parcialmente consumido no model.fit).
        eval_start = time.time()

        val_gen_eval = make_generator(
            TEST_DATAGEN, fd['val'],
            cfg['batch_size'], class_mode='binary',
            label_map=T4_LABEL_MAP, shuffle=False
        )

        y_true, y_pred_prob = [], []
        for imgs, labels in val_gen_eval:
            preds = model.predict(imgs, verbose=0).flatten()
            y_pred_prob.extend(preds)
            y_true.extend(labels.astype(int))
            if len(y_true) >= len(fd['val']):
                break

        eval_duration = time.time() - eval_start

        y_pred = [1 if p >= 0.5 else 0 for p in y_pred_prob]

        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(   y_true, y_pred, zero_division=0)
        acc  = accuracy_score( y_true, y_pred)
        f1   = f1_score(       y_true, y_pred, zero_division=0)

        cm = confusion_matrix(y_true, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            spe = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        else:
            spe = 0.0

        fold_duration  = time.time() - fold_start
        ram_after_eval = get_ram_mb()
        gpu_after_eval = get_gpu_mb()
        ram_sys_after  = get_ram_system_mb()

        r = {
            'fold'       : fold,
            'precision'  : prec,
            'recall'     : rec,
            'accuracy'   : acc,
            'specificity': spe,
            'f1'         : f1,
        }
        results.append(r)

        p = {
            'fold'              : fold,
            'n_train_images'    : len(fd['train']),
            'n_val_images'      : len(fd['val']),
            # 'n_test_images' removido — protocolo 80/20 sem teste separado
            'epochs_run'        : n_epochs_run,
            'best_epoch'        : best_epoch,
            'time_train_s'      : round(train_duration, 2),
            'time_eval_s'       : round(eval_duration, 2),
            'time_fold_total_s' : round(fold_duration, 2),
            'time_train_fmt'    : fmt_duration(train_duration),
            'time_fold_fmt'     : fmt_duration(fold_duration),
            'ram_before_mb'     : round(ram_before, 1),
            'ram_after_train_mb': round(ram_after_train, 1),
            'ram_after_eval_mb' : round(ram_after_eval, 1),
            'ram_delta_mb'      : round(ram_after_eval - ram_before, 1),
            'ram_system_before' : ram_sys_start,
            'ram_system_after'  : ram_sys_after,
            'gpu_before'        : gpu_before,
            'gpu_after_train'   : gpu_after_train,
            'gpu_after_eval'    : gpu_after_eval,
        }
        profiling.append(p)

        history.append({
            'fold'  : fold,
            'epochs': epoch_profiler.epoch_logs,
        })

        print(
            f'  Fold {fold} → Pre={prec:.4f} Rec={rec:.4f} '
            f'Acc={acc:.4f} Spe={spe:.4f} F1={f1:.4f}'
        )
        print(
            f'  Tempo    : treino={fmt_duration(train_duration)} '
            f'| fold total={fmt_duration(fold_duration)}'
        )
        print(
            f'  RAM      : antes={ram_before:.0f} MB '
            f'→ depois={ram_after_eval:.0f} MB '
            f'(Δ={ram_after_eval - ram_before:+.0f} MB)'
        )
        if gpu_after_eval:
            print(
                f'  GPU VRAM : current={gpu_after_eval["current_mb"]:.0f} MB '
                f'| peak={gpu_after_eval["peak_mb"]:.0f} MB'
            )

        tf.keras.backend.clear_session()

    # ── Resumo ────────────────────────────────────────────────────────────────
    model_total_s = time.time() - model_start_t
    model_end_ts  = datetime.datetime.now().isoformat()

    print(f'\n{"─"*50}')
    print(f'Média 5-fold | {architecture.upper()} | Task 4 (80/20)')
    for metric in ['precision', 'recall', 'accuracy', 'specificity', 'f1']:
        vals = [r[metric] for r in results]
        print(f'  {metric:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')

    print(f'\n  Tempo total do modelo : {fmt_duration(model_total_s)}')
    fold_times = [p['time_fold_total_s'] for p in profiling]
    print(f'  Tempo médio por fold  : {fmt_duration(np.mean(fold_times))}')
    ram_deltas = [p['ram_delta_mb'] for p in profiling]
    print(f'  Δ RAM médio por fold  : {np.mean(ram_deltas):+.0f} MB')

    # ── Persistência ─────────────────────────────────────────────────────────
    res_path = os.path.join(model_dir, f'{architecture}_results.json')
    with open(res_path, 'w') as f:
        json.dump(results, f, indent=2)

    profile_summary = {
        'architecture'      : architecture,
        'split_protocol'    : '80_train_20_val_no_test',   # rastreabilidade
        'config'            : cfg,
        'start_timestamp'   : model_start_ts,
        'end_timestamp'     : model_end_ts,
        'total_time_s'      : round(model_total_s, 2),
        'total_time_fmt'    : fmt_duration(model_total_s),
        'mean_fold_time_s'  : round(float(np.mean(fold_times)), 2),
        'mean_fold_time_fmt': fmt_duration(np.mean(fold_times)),
        'folds'             : profiling,
    }
    prof_path = os.path.join(model_dir, f'{architecture}_profiling.json')
    with open(prof_path, 'w') as f:
        json.dump(profile_summary, f, indent=2)

    hist_path = os.path.join(model_dir, f'{architecture}_history.json')
    with open(hist_path, 'w') as f:
        json.dump(history, f, indent=2)

    print(f'\n  {os.path.basename(res_path)}')
    print(f'  {os.path.basename(prof_path)}')
    print(f'  {os.path.basename(hist_path)}')
    return results


print('train_task4 instrumentada carregada.')

### Treinamento para Inceptionv3

In [ ]:
# ── Executar treinamento ─────────────────────────────────────────
results_t4_iv3  = train_task4('inceptionv3')

### Treinamento para Inceptionresnetv2

In [ ]:
# ── Executar treinamento ─────────────────────────────────────────
results_t4_irv2 = train_task4('inceptionresnetv2')

### Treinamento para Xception

In [ ]:
# ── Executar treinamento ─────────────────────────────────────────
results_t4_xception = train_task4('xception')

### Treinamento para Efficientnetv2s

In [ ]:
# ── Executar treinamento ─────────────────────────────────────────
results_t4_effv2s   = train_task4('efficientnetv2s')

## Resumo Consolidado de Profiling (todas as arquiteturas)

In [ ]:
ARCHS     = ['inceptionv3', 'inceptionresnetv2', 'xception', 'efficientnetv2s']
model_dir = os.path.join(MODELS_DIR, 'task4')

rows_perf = []
rows_time = []
rows_mem  = []

for arch in ARCHS:
    res_path  = os.path.join(model_dir, f'{arch}_results.json')
    prof_path = os.path.join(model_dir, f'{arch}_profiling.json')

    if not os.path.exists(res_path):
        print(f'    {arch}: results não encontrado')
        continue

    with open(res_path) as f:
        res = json.load(f)
    df_r = pd.DataFrame(res)

    # Linha de performance
    row_p = {'Arquitetura': arch}
    for m in ['precision','recall','accuracy','specificity','f1']:
        row_p[f'{m}_média'] = round(df_r[m].mean(), 4)
        row_p[f'{m}_desvio padrão']  = round(df_r[m].std(),  4)
    rows_perf.append(row_p)

    if not os.path.exists(prof_path):
        continue

    with open(prof_path) as f:
        prof = json.load(f)

    fold_times = [fd['time_fold_total_s'] for fd in prof['folds']]
    fold_ram   = [fd['ram_delta_mb']      for fd in prof['folds']]
    gpu_peaks  = [
        fd['gpu_after_train']['peak_mb']
        for fd in prof['folds']
        if fd.get('gpu_after_train')
    ]

    rows_time.append({
        'Arquitetura'             : arch,
        'Tempo total'             : prof['total_time_fmt'],
        'Tempo total (s)'         : prof['total_time_s'],
        'Tempo médio/fold'        : prof['mean_fold_time_fmt'],
        'Desvio padrão/fold (s)'  : round(np.std(fold_times), 2),
        'Min fold (s)'            : round(min(fold_times), 0),
        'Max fold (s)'            : round(max(fold_times), 0),
        'Início'                  : prof['start_timestamp'][:19],
        'Fim'                     : prof['end_timestamp'][:19],
    })

    rows_mem.append({
        'Arquitetura'                : arch,
        'Δ RAM médio (MB)'           : round(np.mean(fold_ram), 1),
        'Δ RAM desvio padrão (MB)'   : round(np.std(fold_ram),  1),
        'Δ RAM max (MB)'             : round(max(fold_ram), 1),
        'GPU VRAM peak médio (MB)'   : round(np.mean(gpu_peaks), 1) if gpu_peaks else 'N/A',
        'GPU VRAM peak max (MB)'     : round(max(gpu_peaks), 1)     if gpu_peaks else 'N/A',
    })

print('='*70)
print('TABELA 1 — Métricas de Classificação (μ ± σ dos 5 folds)')
print('='*70)
if rows_perf:
    print(pd.DataFrame(rows_perf).set_index('Arquitetura').to_string())

print('\n' + '='*70)
print('TABELA 2 — Tempo de Execução (T4 - RAM alta)')
print('='*70)
if rows_time:
    print(pd.DataFrame(rows_time).set_index('Arquitetura').to_string())

print('\n' + '='*70)
print('TABELA 3 — Consumo de Memória')
print('='*70)
if rows_mem:
    print(pd.DataFrame(rows_mem).set_index('Arquitetura').to_string())

# Salvar CSV consolidado no Drive
if rows_perf:
    pd.DataFrame(rows_perf).to_csv(
        os.path.join(model_dir, 'summary_performance.csv'), index=False)
if rows_time:
    pd.DataFrame(rows_time).to_csv(
        os.path.join(model_dir, 'summary_timing.csv'), index=False)
if rows_mem:
    pd.DataFrame(rows_mem).to_csv(
        os.path.join(model_dir, 'summary_memory.csv'), index=False)

print('\n CSVs consolidados salvos em models/task4/')

## Célula 13 — Resumo Final dos Resultados

In [ ]:
def print_results_table(results, task_name, arch_name):
    df = pd.DataFrame(results)
    print(f'\n{"─"*65}')
    print(f'{task_name} | {arch_name}')
    print(df[['fold','precision','recall','accuracy','specificity','f1']].to_string(index=False))
    print('  Médias:')
    for col in ['precision','recall','accuracy','specificity','f1']:
        vals = df[col].values
        print(f'    {col:12s}: {vals.mean():.4f} \u00b1 {vals.std():.4f}')


print('='*65)
print('RESUMO FINAL')
print('='*65)

# Resultados Task 4
try:
    print_results_table(results_t4_iv3,  'Task 4 (Cáries)', 'Inception-v3')
    print_results_table(results_t4_irv2, 'Task 4 (Cáries)', 'InceptionResNet-v2')
    print_results_table(results_t4_xception, 'Task 4 (Cáries)', 'Xception')
    print_results_table(results_t4_effv2s, 'Task 4 (Cáries)', 'EfficientNetV2S')
except NameError:
    for arch in ['inceptionv3', 'inceptionresnetv2', 'xception', 'efficientnetv2s']:
        p = os.path.join(MODELS_DIR, 'task4', f'{arch}_results.json')
        if os.path.exists(p):
            with open(p) as f:
                print_results_table(json.load(f), 'Task 4 (Cáries)', arch)